# Day 42：LangGraph 进阶（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-042` ｜ **开发分支**：`feat/day42-langgraph-advanced-writer`  
> **配套 HTML5 课件**：`courseware/day42/index.html` ｜ **独立版本快照**：`versions/day42/`

---

## 🎙️ 老陈开场旁白

早啊！昨天林产品提的那个“总监 2 小时后才在钉钉点同意”的问题，今天老陈带大家一次性彻底拿下。今天四个核心武器全是真金白银的生产硬技能：第一，SQLite Checkpointer 持久化，进程重启都不丢进度；第二，Human-in-the-loop 中断与恢复；第三，节点级自动重试（RetryPolicy）；第四，用 Subgraph 子图 + Fan-out/Fan-in 并发架构做一个“大纲 -> 并行写多章节 -> 汇总润色”的技术方案写作 Agent，直接把长文生成耗时砍掉三分之二！

### 承上启下
- **昨日基线**：昨天（Day 41）我们手写了 `StateGraph` 引擎并跑通了同步版的 ReAct 和采购审批流，但留下了生产级异步中断、状态持久化以及多任务并发加速三大进阶课题。
- **明日预告**：明天（Day 43）我们将从单 Agent 状态图跃迁到“多 Agent 协同系统”，学习 Supervisor 模式、协作模式与层级模式，并组建星辰数科“搜索员+分析师+撰写员”研究团队。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day42'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-042）

星辰数科售前方案团队每周要为政企客户撰写 3000 字以上的技术方案书。以往串行生成 4 个章节耗时长达 40 秒，且生成前需要售前经理确认大纲，一旦中途 API 抖动报错，之前写好的章节全部作废重来。

**今日迭代目标：**
- 实现 `SQLiteCheckpointer`，按 `thread_id` 持久化每一步状态快照与下一待执行节点，支持 `interrupt_before` 挂起与 `resume` 恢复。
- 为状态图节点增加可配置的指数退避错误重试机制（`RetryPolicy`）。
- 基于 Subgraph（单章撰写+质量校验子图）与 Fan-out/Fan-in 并行分发，实现“大纲规划 -> 并发写章节 -> 汇总润色”写作 Agent。

**验收标准：**
- 在 `interrupt_before=['human_review']` 处自动暂停并写入 SQLite，模拟新进程重启后仍能按 `thread_id` 注入人工修改意见并恢复执行至 `END`。
- 模拟某节点前 2 次抛临时网络异常、第 3 次成功时，重试机制可自动恢复且最终状态无误。
- 写作 Agent 并行执行 3 个章节子图的总耗时明显低于串行总耗时，且最终成文严格保持大纲章节顺序。

---

## 📚 课堂核心技术笔记

### 1. 1. 真实 LangGraph Checkpointer 与 HITL 生产配置

生产环境中切勿使用内存版 `MemorySaver`（多副本 K8s 部署或服务重启即丢失），应使用 `langgraph-checkpoint-postgres`：
```python
from langgraph.checkpoint.sqlite import SqliteSaver
with SqliteSaver.from_conn_string('checkpoints.db') as checkpointer:
    app = builder.compile(checkpointer=checkpointer, interrupt_before=['human_review'])
    config = {'configurable': {'thread_id': 'order-2026-001'}}
    app.invoke(init_state, config=config)  # 自动停在 human_review 之前
    app.update_state(config, {'approved': True})  # 人工审批后注入状态
    app.invoke(None, config=config)  # 传入 None 从断点恢复执行
```

> **⚠️ 老陈避坑**：坑点：`State` 里存放的所有字段必须支持 JSON/Msgpack 序列化！千万别把数据库连接对象、HTTP Client 或线程锁塞进 `State`，否则 Checkpointer 落库必报错。

### 2. 2. Fan-out/Fan-in 并行节点的顺序一致性保障

在并发撰写 4 个技术方案章节时，各章节耗时不同，谁先返回是不确定的。正确姿势是每个并发子任务返回结构化对象 `{'index': idx, 'title': title, 'body': text}`，Fan-in 汇聚节点执行 `sorted(chapters, key=lambda c: c['index'])` 后再做全文润色。

> **⚠️ 老陈避坑**：坑点：如果在并行分支里直接修改共享外部文件或共享全局列表，极易引发竞态条件；每个子图必须只返回局部纯数据，由引擎安全归约。

### 3. 3. 区分“可重试瞬时错误”与“不可重试业务错误”

网络超时、HTTP 429 限流、502 网关错误属于瞬时故障，适合用 `RetryPolicy` 自动重试；而参数校验错误 `ValueError`、权限不足 `PermissionError` 重试 100 次还是会错，应该立刻快速失败（Fail Fast），避免白白浪费时间。

> **⚠️ 老陈避坑**：坑点：重试只适合幂等节点（如查库、调 LLM 生成文本）；对于“扣款”、“发真实邮件”等非幂等写操作，重试前必须具备幂等键（Idempotency Key）。


## 💻 实操步骤 1：步骤一：实现 SQLite Checkpointer、HITL 中断恢复与节点重试引擎

- **目标文件**：`versions/day42/src/day42_checkpoint_hitl.py`
- **老陈代码旁白**：现在我们把昨天的 `StateGraph` 全面升级为生产级 `PersistentGraph`：接入真实 `sqlite3` 持久化每一步快照，支持 `interrupt_before` 挂起和 `resume(thread_id, patch)` 跨实例恢复，并内置指数退避 `RetryPolicy`。


In [ ]:
"""Day 42 - 步骤一：SQLite Checkpointer 状态持久化、HITL 中断恢复与节点重试机制。"""

from __future__ import annotations

import copy
import json
import sqlite3
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple, Type

START = "__start__"
END = "__end__"


@dataclass
class RetryPolicy:
    """节点级错误重试策略（对齐 LangGraph RetryPolicy）。"""

    max_attempts: int = 3
    initial_delay_sec: float = 0.01
    backoff_factor: float = 2.0
    retry_on: Tuple[Type[Exception], ...] = (ConnectionError, TimeoutError, RuntimeError)


class SQLiteCheckpointer:
    """基于 SQLite 的状态快照持久化存储，支持断点续跑与历史回溯。"""

    def __init__(self, db_path: str | Path = ":memory:") -> None:
        self.db_path = str(db_path)
        self._conn = sqlite3.connect(self.db_path, check_same_thread=False)
        self._init_table()

    def _init_table(self) -> None:
        self._conn.execute(
            """
            CREATE TABLE IF NOT EXISTS checkpoints (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                thread_id TEXT NOT NULL,
                step_no INTEGER NOT NULL,
                next_node TEXT NOT NULL,
                status TEXT NOT NULL,
                state_json TEXT NOT NULL,
                created_at REAL NOT NULL
            )
            """
        )
        self._conn.commit()

    def save(self, thread_id: str, step_no: int, next_node: str, status: str, state: Dict[str, Any]) -> None:
        self._conn.execute(
            "INSERT INTO checkpoints (thread_id, step_no, next_node, status, state_json, created_at) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            (thread_id, step_no, next_node, status, json.dumps(state, ensure_ascii=False), time.time()),
        )
        self._conn.commit()

    def load_latest(self, thread_id: str) -> Optional[Dict[str, Any]]:
        row = self._conn.execute(
            "SELECT step_no, next_node, status, state_json FROM checkpoints "
            "WHERE thread_id = ? ORDER BY id DESC LIMIT 1",
            (thread_id,),
        ).fetchone()
        if not row:
            return None
        return {
            "step_no": row[0],
            "next_node": row[1],
            "status": row[2],
            "state": json.loads(row[3]),
        }

    def get_history(self, thread_id: str) -> List[Dict[str, Any]]:
        rows = self._conn.execute(
            "SELECT step_no, next_node, status, state_json FROM checkpoints "
            "WHERE thread_id = ? ORDER BY id ASC",
            (thread_id,),
        ).fetchall()
        return [
            {"step_no": r[0], "next_node": r[1], "status": r[2], "state": json.loads(r[3])}
            for r in rows
        ]

    def close(self) -> None:
        self._conn.close()


class PersistentGraph:
    """支持 Checkpointer、HITL 中断恢复与节点重试的进阶状态图引擎。"""

    def __init__(
        self,
        checkpointer: SQLiteCheckpointer,
        interrupt_before: Optional[List[str]] = None,
    ) -> None:
        self.checkpointer = checkpointer
        self.interrupt_before = set(interrupt_before or [])
        self._nodes: Dict[str, Tuple[Callable[[Dict[str, Any]], Dict[str, Any]], Optional[RetryPolicy]]] = {}
        self._edges: Dict[str, str] = {}

    def add_node(
        self,
        name: str,
        func: Callable[[Dict[str, Any]], Dict[str, Any]],
        retry_policy: Optional[RetryPolicy] = None,
    ) -> "PersistentGraph":
        self._nodes[name] = (func, retry_policy)
        return self

    def add_edge(self, src: str, dst: str) -> "PersistentGraph":
        self._edges[src] = dst
        return self

    def _run_node_with_retry(self, name: str, state: Dict[str, Any]) -> Dict[str, Any]:
        func, policy = self._nodes[name]
        if policy is None:
            return func(copy.deepcopy(state))

        delay = policy.initial_delay_sec
        last_exc: Optional[Exception] = None
        for attempt in range(1, policy.max_attempts + 1):
            try:
                return func(copy.deepcopy(state))
            except policy.retry_on as exc:
                last_exc = exc
                if attempt < policy.max_attempts:
                    time.sleep(delay)
                    delay *= policy.backoff_factor
        assert last_exc is not None
        raise last_exc

    def run(self, thread_id: str, initial_state: Dict[str, Any]) -> Dict[str, Any]:
        """启动新线程任务，若命中 interrupt_before 则落库挂起并返回 INTERRUPTED。"""
        state = copy.deepcopy(initial_state)
        next_node = self._edges[START]
        return self._drive(thread_id=thread_id, state=state, current=next_node, step_no=0, resuming=False)

    def resume(self, thread_id: str, human_patch: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """从 SQLite 加载最新断点，合并人工输入补丁，继续执行后续节点。"""
        ckpt = self.checkpointer.load_latest(thread_id)
        if not ckpt:
            raise ValueError(f"未找到 thread_id='{thread_id}' 的持久化快照")
        if ckpt["status"] != "INTERRUPTED":
            raise ValueError(f"当前状态为 {ckpt['status']}，无需恢复")

        state = ckpt["state"]
        if human_patch:
            state.update(human_patch)
        return self._drive(
            thread_id=thread_id,
            state=state,
            current=ckpt["next_node"],
            step_no=ckpt["step_no"],
            resuming=True,
        )

    def _drive(
        self,
        thread_id: str,
        state: Dict[str, Any],
        current: str,
        step_no: int,
        resuming: bool,
    ) -> Dict[str, Any]:
        first_node = True
        while current != END:
            # 命中人工中断点（恢复执行时的首个节点放行）
            if current in self.interrupt_before and not (resuming and first_node):
                self.checkpointer.save(thread_id, step_no, current, "INTERRUPTED", state)
                return {"status": "INTERRUPTED", "waiting_node": current, "state": state}

            first_node = False
            update = self._run_node_with_retry(current, state)
            state.update(update)
            step_no += 1
            current = self._edges[current]
            self.checkpointer.save(thread_id, step_no, current, "RUNNING" if current != END else "COMPLETED", state)

        return {"status": "COMPLETED", "waiting_node": None, "state": state}


if __name__ == "__main__":
    ckpt = SQLiteCheckpointer(":memory:")
    g = PersistentGraph(checkpointer=ckpt, interrupt_before=["publish"])
    g.add_node("draft", lambda s: {"outline": [f"1.{s['topic']}背景", f"2.{s['topic']}架构"]})
    g.add_node("publish", lambda s: {"published": True, "reviewer": s.get("reviewer", "未知")})
    g.add_edge(START, "draft")
    g.add_edge("draft", "publish")
    g.add_edge("publish", END)

    r1 = g.run("doc-001", {"topic": "星辰智脑中台"})
    print("第一次执行状态:", r1["status"], "| 等待节点:", r1["waiting_node"])
    r2 = g.resume("doc-001", human_patch={"reviewer": "林产品", "approved": True})
    print("人工放行后状态:", r2["status"], "| 最终状态:", r2["state"])


### 🔍 步骤 1 代码深度解析

1. `SQLiteCheckpointer` 每次推进节点后都持久化一份完整快照，既支持进程崩溃后无缝恢复，也能通过 `get_history(thread_id)` 审计全过程。
2. `interrupt_before=['publish']` 让图在真正执行敏感发布动作前主动刹车，返回 `INTERRUPTED` 给上层 API；待人工在审批页点击通过后，调用 `resume(thread_id, human_patch=...)` 即可精准续跑。
3. `_run_node_with_retry` 仅捕获 `policy.retry_on` 中声明的瞬时异常类型，并按指数退避休眠重试。

**预期运行输出参考：**
```text
第一次执行状态: INTERRUPTED | 等待节点: publish
人工放行后状态: COMPLETED | 最终状态: {'topic': '星辰智脑中台', 'outline': ['1.星辰智脑中台背景', '2.星辰智脑中台架构'], 'reviewer': '林产品', 'approved': True, 'published': True}
```


## 💻 实操步骤 2：步骤二：Subgraph 子图 + Fan-out/Fan-in 并行技术方案写作 Agent

- **目标文件**：`versions/day42/src/day42_parallel_writer.py`
- **老陈代码旁白**：接下来我们把 Subgraph 和 Fan-out/Fan-in 结合起来，打造星辰数科售前团队的“并行技术方案写作 Agent”：先生成 3 章大纲，再用 `ThreadPoolExecutor` 把 3 个章节同时派发给「起草 -> 质检」子图并行撰写，最后按原始章节序号汇总润色成完整 Markdown 方案！


In [ ]:
"""Day 42 - 步骤二：Subgraph 子图 + Fan-out/Fan-in 并行写作 Agent 实战。"""

from __future__ import annotations

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from typing import Any, Dict, List


@dataclass
class ChapterDraft:
    index: int
    title: str
    content: str
    passed_qa: bool
    revisions: int


class ChapterWriterSubgraph:
    """
    独立可复用的单章写作 Subgraph：
    节点流转：draft_section -> qa_check -> (不合格则 revise_section -> qa_check) -> END
    """

    def __init__(self, simulated_io_sec: float = 0.05) -> None:
        self.simulated_io_sec = simulated_io_sec

    def invoke(self, section_spec: Dict[str, Any]) -> ChapterDraft:
        idx = section_spec["index"]
        title = section_spec["title"]
        topic = section_spec["topic"]

        # 节点 1：起草初稿（模拟外部 LLM 调用耗时）
        time.sleep(self.simulated_io_sec)
        body = f"针对《{topic}》的「{title}」，星辰数科采用模块化微服务与高可用状态机架构，保障核心链路 99.95% 可用性。"
        revisions = 0

        # 节点 2：质检循环（检查是否包含必要企业落地要素）
        while "量化指标" not in body and revisions < 2:
            revisions += 1
            body += " 量化指标：端到端 P95 延迟低于 1.5 秒，自动化单测覆盖率不低于 85%。"

        return ChapterDraft(
            index=idx,
            title=title,
            content=body,
            passed_qa=("量化指标" in body),
            revisions=revisions,
        )


class ParallelProposalWriterAgent:
    """
    主图编排：
    outline_node -> fan_out_chapters (并行调用 ChapterWriterSubgraph) -> fan_in_polish_node
    """

    def __init__(self, max_workers: int = 4, simulated_io_sec: float = 0.05) -> None:
        self.max_workers = max_workers
        self.chapter_subgraph = ChapterWriterSubgraph(simulated_io_sec=simulated_io_sec)

    def _outline_node(self, topic: str) -> List[Dict[str, Any]]:
        return [
            {"index": 1, "title": "一、项目背景与痛点分析", "topic": topic},
            {"index": 2, "title": "二、核心系统架构与技术选型", "topic": topic},
            {"index": 3, "title": "三、安全合规与交付验收计划", "topic": topic},
        ]

    def _fan_out_write_chapters(self, outline: List[Dict[str, Any]]) -> List[ChapterDraft]:
        """Fan-out 并行执行所有章节子图，并在 Fan-in 时按原始 index 恢复严格顺序。"""
        results: List[ChapterDraft] = []
        with ThreadPoolExecutor(max_workers=self.max_workers) as pool:
            futures = [pool.submit(self.chapter_subgraph.invoke, sec) for sec in outline]
            for fut in as_completed(futures):
                results.append(fut.result())
        # 关键：按原始大纲序号排序，防止并行乱序
        results.sort(key=lambda ch: ch.index)
        return results

    def _polish_node(self, topic: str, chapters: List[ChapterDraft]) -> str:
        sections_md = [f"# {topic} —— 技术落地方案\n"]
        for ch in chapters:
            sections_md.append(f"## {ch.title}\n{ch.content}\n")
        sections_md.append("---\n*本文档由星辰智脑并行写作 Agent 自动生成并通过质检子图校验。*")
        return "\n".join(sections_md)

    def run(self, topic: str) -> Dict[str, Any]:
        t0 = time.perf_counter()
        outline = self._outline_node(topic)
        chapters = self._fan_out_write_chapters(outline)
        final_doc = self._polish_node(topic, chapters)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 2)
        return {
            "topic": topic,
            "chapter_count": len(chapters),
            "all_passed_qa": all(c.passed_qa for c in chapters),
            "ordered_titles": [c.title for c in chapters],
            "elapsed_ms": elapsed_ms,
            "document": final_doc,
        }


if __name__ == "__main__":
    writer = ParallelProposalWriterAgent(max_workers=3, simulated_io_sec=0.05)
    out = writer.run("星辰数科智能客服升级项目")
    print(f"并行生成 {out['chapter_count']} 章耗时: {out['elapsed_ms']} ms | 全部质检通过: {out['all_passed_qa']}")
    print(out["document"])


### 🔍 步骤 2 代码深度解析

1. `ChapterWriterSubgraph` 封装了“初稿起草 -> 质检 -> 自动补齐量化指标”的微循环，主图无需关心单章内部校验细节。
2. `_fan_out_write_chapters` 使用 `ThreadPoolExecutor` 并发派发 3 个子图，模拟单章耗时 50ms 时，3 章并行总耗时仅约 55ms（而非串行 150ms）。
3. `results.sort(key=lambda ch: ch.index)` 是 Fan-in 汇总的黄金法则，确保无论哪一章先写完，最终文档结构永远井然有序。

**预期运行输出参考：**
```text
并行生成 3 章耗时: 52.14 ms | 全部质检通过: True
# 星辰数科智能客服升级项目 —— 技术落地方案

## 一、项目背景与痛点分析
针对《星辰数科智能客服升级项目》的「一、项目背景与痛点分析」，星辰数科采用模块化微服务与高可用状态机架构，保障核心链路 99.95% 可用性。 量化指标：端到端 P95 延迟低于 1.5 秒，自动化单测覆盖率不低于 85%。
...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day42/tests/test_day42.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 42 自动化单元测试：验证 SQLite 跨实例断点续跑、节点重试与并行写作 Agent。"""

from pathlib import Path
import pytest
from src.day42_checkpoint_hitl import (
    PersistentGraph,
    RetryPolicy,
    SQLiteCheckpointer,
    START,
    END,
)
from src.day42_parallel_writer import ParallelProposalWriterAgent


def test_sqlite_checkpointer_hitl_across_instances(tmp_path: Path):
    db_file = tmp_path / "starmind_ckpt.db"
    # 实例 1：执行到 publish 前中断落库
    ckpt1 = SQLiteCheckpointer(db_file)
    g1 = PersistentGraph(checkpointer=ckpt1, interrupt_before=["publish"])
    g1.add_node("draft", lambda s: {"draft_text": f"方案:{s['topic']}"})
    g1.add_node("publish", lambda s: {"published": True, "approver": s["approver"]})
    g1.add_edge(START, "draft")
    g1.add_edge("draft", "publish")
    g1.add_edge("publish", END)

    res1 = g1.run("thread-99", {"topic": "金融风控Agent"})
    assert res1["status"] == "INTERRUPTED"
    assert res1["waiting_node"] == "publish"
    ckpt1.close()

    # 实例 2：模拟服务重启后重新打开同一个 SQLite 文件并恢复执行
    ckpt2 = SQLiteCheckpointer(db_file)
    g2 = PersistentGraph(checkpointer=ckpt2, interrupt_before=["publish"])
    g2.add_node("draft", lambda s: {"draft_text": f"方案:{s['topic']}"})
    g2.add_node("publish", lambda s: {"published": True, "approver": s["approver"]})
    g2.add_edge(START, "draft")
    g2.add_edge("draft", "publish")
    g2.add_edge("publish", END)

    res2 = g2.resume("thread-99", human_patch={"approver": "老陈"})
    assert res2["status"] == "COMPLETED"
    assert res2["state"]["published"] is True
    assert res2["state"]["approver"] == "老陈"
    assert len(ckpt2.get_history("thread-99")) >= 3
    ckpt2.close()


def test_node_retry_policy_recovers_transient_error():
    attempts = {"count": 0}

    def flaky_api_node(_state):
        attempts["count"] += 1
        if attempts["count"] < 3:
            raise ConnectionError("模拟临时 502 抖动")
        return {"ok": True}

    ckpt = SQLiteCheckpointer(":memory:")
    g = PersistentGraph(checkpointer=ckpt)
    g.add_node("flaky", flaky_api_node, retry_policy=RetryPolicy(max_attempts=3, initial_delay_sec=0.001))
    g.add_edge(START, "flaky")
    g.add_edge("flaky", END)

    res = g.run("retry-1", {})
    assert res["status"] == "COMPLETED"
    assert res["state"]["ok"] is True
    assert attempts["count"] == 3
    ckpt.close()


def test_parallel_writer_order_and_speedup():
    writer = ParallelProposalWriterAgent(max_workers=3, simulated_io_sec=0.04)
    out = writer.run("星辰政企知识库")
    assert out["chapter_count"] == 3
    assert out["all_passed_qa"] is True
    assert out["ordered_titles"] == [
        "一、项目背景与痛点分析",
        "二、核心系统架构与技术选型",
        "三、安全合规与交付验收计划",
    ]
    # 3 章各 40ms 并行总耗时应显著低于串行 120ms
    assert out["elapsed_ms"] < 115

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 Checkpointer 实现时间旅行回滚方法

**题目要求：**
为 `SQLiteCheckpointer` 编写一个 `load_step(thread_id, step_no)` 查询函数，允许开发者按指定步骤号读取历史快照，实现状态“时间旅行”。

**思路提示：** 在 `checkpoints` 表上增加 `WHERE thread_id = ? AND step_no = ?` 条件查询。


In [ ]:
"""Day 42 作业 1 参考答案：Checkpointer 时间旅行按步回溯。"""

import json
import sqlite3


def load_step(conn: sqlite3.Connection, thread_id: str, step_no: int) -> dict | None:
    row = conn.execute(
        "SELECT next_node, status, state_json FROM checkpoints WHERE thread_id = ? AND step_no = ? LIMIT 1",
        (thread_id, step_no),
    ).fetchone()
    if not row:
        return None
    return {"next_node": row[0], "status": row[1], "state": json.loads(row[2])}


if __name__ == "__main__":
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE checkpoints (thread_id TEXT, step_no INT, next_node TEXT, status TEXT, state_json TEXT)")
    conn.execute("INSERT INTO checkpoints VALUES ('t1', 1, 'b', 'RUNNING', '{\"step\": 1}')")
    conn.execute("INSERT INTO checkpoints VALUES ('t1', 2, '__end__', 'COMPLETED', '{\"step\": 2}')")
    snap = load_step(conn, "t1", 1)
    print("回溯到第 1 步:", snap)
    assert snap["state"]["step"] == 1


**答案解析与评分标准：**

时间旅行（Time Travel）是 LangGraph 调试线上复杂 Agent 故障的神器：无需从头重跑耗时节点，直接从出错前一步分叉重演。


### 作业 2（企业实战）：实现带降级兜底的并行章节 Fan-out

**题目要求：**
如果并行写作的 3 个章节中有 1 个章节因模型超时抛异常，要求其余 2 个章节正常保留，失败章节填充“【待补充】章节生成超时”占位提示，不要让整个方案书全盘崩溃。

**思路提示：** 在捕获 `fut.result()` 时用 `try/except Exception` 包装，并为失败任务构造兜底 `ChapterDraft`。


In [ ]:
"""Day 42 作业 2 参考答案：部分失败容错的并行 Fan-out 执行器。"""

from concurrent.futures import ThreadPoolExecutor


def write_one(sec: dict) -> dict:
    if sec["index"] == 2:
        raise TimeoutError("第2章接口超时")
    return {"index": sec["index"], "text": f"正文-{sec['title']}", "ok": True}


def safe_parallel_write(sections: list[dict]) -> list[dict]:
    out = []
    with ThreadPoolExecutor(max_workers=3) as pool:
        future_map = {pool.submit(write_one, s): s for s in sections}
        for fut, sec in future_map.items():
            try:
                out.append(fut.result())
            except Exception as exc:
                out.append({"index": sec["index"], "text": f"【降级占位】生成失败: {exc}", "ok": False})
    out.sort(key=lambda x: x["index"])
    return out


if __name__ == "__main__":
    secs = [{"index": 1, "title": "背景"}, {"index": 2, "title": "架构"}, {"index": 3, "title": "计划"}]
    res = safe_parallel_write(secs)
    print(res)
    assert res[0]["ok"] is True and res[1]["ok"] is False and res[2]["ok"] is True


**答案解析与评分标准：**

在并发扇出场景中，采用“部分成功 + 局部降级”比“一损俱损全盘失败”能显著提升 B 端用户满意度。


### 作业 3（架构进阶）：为 HITL 中断增加超时自动过期清理策略

**题目要求：**
很多审批单挂起 7 天没人理会，一直占着待办列表。请编写 `expire_stale_threads(conn, ttl_seconds)`，将超过 TTL 仍处于 `INTERRUPTED` 状态的最新会话标记为 `EXPIRED`。

**思路提示：** 查询每个 `thread_id` 最新的 `INTERRUPTED` 记录，若 `now - created_at > ttl_seconds` 则更新状态为 `EXPIRED`。


In [ ]:
"""Day 42 作业 3 参考答案：HITL 挂起会话超时过期清理。"""

import sqlite3
import time


def expire_stale_threads(conn: sqlite3.Connection, ttl_seconds: float, now: float | None = None) -> int:
    ts = now if now is not None else time.time()
    cutoff = ts - ttl_seconds
    cur = conn.execute(
        "UPDATE checkpoints SET status = 'EXPIRED' WHERE status = 'INTERRUPTED' AND created_at < ?",
        (cutoff,),
    )
    conn.commit()
    return cur.rowcount


if __name__ == "__main__":
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE checkpoints (thread_id TEXT, status TEXT, created_at REAL)")
    conn.execute("INSERT INTO checkpoints VALUES ('old-1', 'INTERRUPTED', 100.0)")
    conn.execute("INSERT INTO checkpoints VALUES ('new-1', 'INTERRUPTED', 990.0)")
    expired = expire_stale_threads(conn, ttl_seconds=50.0, now=1000.0)
    print("过期清理数量:", expired)
    assert expired == 1


**答案解析与评分标准：**

持久化挂起任务必须配套 TTL 过期策略与定时巡检任务，防止数据库长期堆积僵尸会话。


---

## 🎯 老陈收尾复盘

漂亮！今天我们把 LangGraph 最硬核的四个进阶特性全部吃透落地了。记住：高可用 Agent 的秘诀不是祈祷大模型和网络永远不出错，而是靠 Checkpointer 兜底状态、靠 RetryPolicy 扛住瞬时抖动、靠 Fan-out/Fan-in 压榨并发延迟。周末前把代码整理好，明天进军多 Agent 团队协作！
